# 01 — Data Check + Baseline TSMOM Replication

Download FF49 + 1M Tbill from Ken French Data Library. Run static
`(slow=12, w_fast=0)` 12-month TSMOM as a sanity check. Confirm the
order-of-magnitude Sharpe matches Moskowitz–Ooi–Pedersen (2012).

This notebook is **not** part of the headline OOS test. It is a pre-OOS
data sanity check.


In [ ]:
import sys, os
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trend import (fetch_ff49, fetch_tbill_1m,
                    backtest_spec, summarize, sharpe_annualized,
                    plot_equity_with_drawdown)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})


## 1. Data download and coverage check

In [ ]:
ff49 = fetch_ff49(cache_dir="../data")
rf   = fetch_tbill_1m(cache_dir="../data")
print(f"FF49: {ff49.shape},  range {ff49.index.min().date()} ~ {ff49.index.max().date()}")
print(f"Tbill: n={len(rf)}, mean {rf.mean()*12*100:.2f}% / yr")

# Per-industry first-valid date — staggered universe
fv = ff49.apply(lambda c: c.first_valid_index())
print(f"\nFirst-valid date distribution:")
print(fv.value_counts().sort_index().head())
print(f"\nMonths with all 49 industries valid: {(ff49.notna().all(axis=1)).sum()} / {len(ff49)}")


## 2. Baseline TSMOM (slow=12, w_fast=0)

In [ ]:
res = backtest_spec(
    ff49, rf=rf,
    slow=12, fast=2, w_slow=1.0, w_fast=0.0,
    asset_vol_target=0.10, asset_vol_lookback=36,
    port_vol_target=0.10, port_vol_lookback=36,
    port_lev_cap=3.0, tcost_bps=20.0,
)
print(f"Effective sample: {res['gross_return'].dropna().index.min().date()} ~ {res['gross_return'].dropna().index.max().date()}")
print(f"Avg leverage: {res['leverage'].mean():.2f}x  (cap-hit: {(res['leverage']>=2.99).mean()*100:.1f}%)")
print(f"Avg monthly turnover: {res['turnover'].mean()*100:.2f}%")


In [ ]:
# Summary stats — full sample and OOS sub-period
summary = pd.DataFrame({
    "Full (1929-)": pd.concat([
        summarize(res["gross_return"]).rename("gross"),
        summarize(res["net_return"]).rename("net"),
    ], axis=1).T,
    "OOS (1976-)": pd.concat([
        summarize(res["gross_return"].loc["1976-07":]).rename("gross"),
        summarize(res["net_return"].loc["1976-07":]).rename("net"),
    ], axis=1).T,
})
display(summary)


**Sanity check**: MOP (2012) report annualized Sharpe ~0.6+ for
diversified TSMOM on a multi-asset universe. FF49 (US equity industries
only) is a single asset class with high cross-sectional correlation, so
TSMOM here should yield Sharpe in the 0.3–0.5 range — consistent with
literature (e.g., Asness et al on equity industry TSMOM).

## 3. Equity curve sanity check

In [ ]:
plot_equity_with_drawdown(
    {"baseline gross": res["gross_return"],
     "baseline net":   res["net_return"]},
    title=f"Static TSMOM (slow=12, w_fast=0) — full sample",
    log_scale=True,
)
plt.show()


## Notes

- Effective backtest start is 1929-07 (after 36M vol-scaling warmup for
  industries with 1926-07 first-valid date). Some industries enter the
  universe later; they are excluded from the EW portfolio until they have
  enough history.
- This baseline `(w_fast = 0)` is the static TSMOM benchmark used as the
  null reference in `02_walkforward.ipynb`.
